# nf_sweep_v2 Smoke Train And Check

Use this notebook before launching a production sweep. It creates a disposable tiny config from one `nf_sweep_v2` config, trains it for a few epochs, samples a few fields, and runs the standard quick diagnostics.

This is not a production metric. It is meant to catch broken configs, bad normalization, sampler failures, dead EMA paths, and obviously bad P(k)/one-point behavior in minutes instead of waiting for an array job.

In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

PROJECT_DIR = Path('/home/jiamingp/diffusion_models_repo')
COSMODIFF_DIR = Path('/home/jiamingp/Diffusion_model/cosmo_diffusion_main')
PYTHON_BIN = Path('/home/jiamingp/venvs/cosmodiff_nf/bin/python')

BASE_RUN_NAME = 'nf_sweep_v2_u128_n500_e100_nick_default'
SMOKE_TAG = 'smoke_n64_e3'
SMOKE_NAME = f'{BASE_RUN_NAME}_{SMOKE_TAG}'

BASE_CONFIG = PROJECT_DIR / 'local/nf_sweep_v2/configs' / f'{BASE_RUN_NAME}.yaml'
SMOKE_DIR = PROJECT_DIR / 'local/nf_sweep_v2_smoke'
SMOKE_CONFIG_DIR = SMOKE_DIR / 'configs'
SMOKE_CHECKPOINT_ROOT = Path('/scratch/huterer_root/huterer0/jiamingp/saved_runs/nf_sweep_v2_smoke')
SMOKE_CHECKPOINT_DIR = SMOKE_CHECKPOINT_ROOT / f'{SMOKE_NAME}_checkpoints'
SMOKE_SAMPLE_ROOT = PROJECT_DIR / 'results/nf_sweep_v2_smoke/samples'
SMOKE_OUTPUT_DIR = PROJECT_DIR / 'results/nf_sweep_v2_smoke/quickcheck'
SMOKE_CONFIG = SMOKE_CONFIG_DIR / f'{SMOKE_NAME}.yaml'
SMOKE_SAMPLE_PATH = SMOKE_SAMPLE_ROOT / f'{SMOKE_NAME}_seed123_raw_train_full.npy'

RUN_TRAIN = True
RUN_SAMPLE = True
OVERWRITE_CONFIG = True

# Small enough for interactive debugging. Increase only after the path works.
SMOKE_RAW_CUBES = 64
SMOKE_EPOCHS = 3
SMOKE_BATCH_SIZE = 32
SMOKE_EMA_SIGMA_RELS = [0.05, 0.25]
SMOKE_EMA_BURN_IN = 0
SMOKE_N_GENERATE = 8
SMOKE_SAMPLE_BATCH = 4
SEED = 123
DEVICE = 'cuda'

# Keep None to use the production train scheduler. Set 100 for an even faster code-path check.
SMOKE_NUM_TRAIN_TIMESTEPS = None

MAX_REAL_CUBES_FOR_PLOTS = 8
MAX_REAL_HIST = 256
MAX_REAL_PK = 128
PK_NBINS = 25

for d in [SMOKE_CONFIG_DIR, SMOKE_SAMPLE_ROOT, SMOKE_OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('base config:', BASE_CONFIG)
print('smoke config:', SMOKE_CONFIG)
print('smoke checkpoints:', SMOKE_CHECKPOINT_DIR)
print('smoke sample:', SMOKE_SAMPLE_PATH)

In [ ]:
def cosmodiff_env() -> dict[str, str]:
    env = os.environ.copy()
    stub_root = PROJECT_DIR / 'results/cache/python_stubs'
    metrics_dir = stub_root / 'sklearn/metrics'
    metrics_dir.mkdir(parents=True, exist_ok=True)
    (stub_root / 'sklearn/__init__.py').write_text('from . import metrics\n')
    (metrics_dir / '__init__.py').write_text(
        "def roc_curve(*args, **kwargs):\n"
        "    raise RuntimeError('sklearn.metrics.roc_curve is stubbed for cosmodiff smoke tests')\n"
    )
    env['COSMODIFF_DIR'] = str(COSMODIFF_DIR)
    env['COSMODIFF_STUB_SKLEARN'] = '1'
    env['TORCHDYNAMO_DISABLE'] = '1'
    env['PYTHONPATH'] = ':'.join([
        str(stub_root),
        str(COSMODIFF_DIR),
        str(PROJECT_DIR),
        env.get('PYTHONPATH', ''),
    ])
    return env


def run_cmd(cmd: list[str], *, check: bool = True) -> subprocess.CompletedProcess:
    print('Running:', ' '.join(shlex.quote(str(x)) for x in cmd), flush=True)
    return subprocess.run([str(x) for x in cmd], cwd=PROJECT_DIR, env=cosmodiff_env(), check=check)


check_code = '\n'.join([
    'import inspect',
    'import os',
    'from pathlib import Path',
    'from cosmodiff import optim',
    'from cosmodiff.transform import Transform',
    "root = Path(os.environ['COSMODIFF_DIR']).resolve()",
    'optim_path = Path(inspect.getsourcefile(optim)).resolve()',
    "assert root in optim_path.parents, f'wrong cosmodiff import: {optim_path}, expected under {root}'",
    "print('OK cosmodiff:', optim_path)",
    "print('Transform:', Transform)",
    "print('train args include EMA:', {'ema_sigma_rels', 'ema_burn_in'}.issubset(inspect.signature(optim.train).parameters))",
])
run_cmd([PYTHON_BIN, '-c', check_code])

In [ ]:
if not BASE_CONFIG.exists():
    raise FileNotFoundError(BASE_CONFIG)

with BASE_CONFIG.open() as f:
    cfg: dict[str, Any] = yaml.safe_load(f)

cfg = json.loads(json.dumps(cfg))  # plain deep copy
cfg['io']['output_dir'] = str(SMOKE_CHECKPOINT_DIR)
cfg['data']['n_samples'] = SMOKE_RAW_CUBES
cfg['data']['seed'] = cfg['data'].get('seed', None)

cfg['train']['num_epochs'] = SMOKE_EPOCHS
cfg['train']['batch_size'] = SMOKE_BATCH_SIZE
cfg['train']['checkpoint_every_n_epochs'] = 1
cfg['train']['dataloader_num_workers'] = 0
cfg['train']['ema_sigma_rels'] = SMOKE_EMA_SIGMA_RELS
cfg['train']['ema_burn_in'] = SMOKE_EMA_BURN_IN
cfg['train']['ema_update_every'] = 1
cfg['train']['verbose'] = True

if SMOKE_NUM_TRAIN_TIMESTEPS is not None:
    cfg['noise_scheduler']['kwargs']['num_train_timesteps'] = int(SMOKE_NUM_TRAIN_TIMESTEPS)

cfg.setdefault('generate', {})
cfg['generate']['n_samples'] = SMOKE_N_GENERATE
cfg['generate']['batch_size'] = SMOKE_SAMPLE_BATCH
cfg['generate']['num_steps'] = None
cfg['generate']['scheduler'] = None
cfg['generate']['seed'] = SEED
cfg['generate']['ema_sigma_rel'] = None

if OVERWRITE_CONFIG or not SMOKE_CONFIG.exists():
    with SMOKE_CONFIG.open('w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

print(SMOKE_CONFIG.read_text())

In [ ]:
if RUN_TRAIN:
    run_cmd([
        PYTHON_BIN,
        'scripts/train_cosmodiff.py',
        '--config',
        SMOKE_CONFIG,
        '--cosmodiff-train',
        COSMODIFF_DIR / 'scripts/cosmodiff_train.py',
    ])
else:
    print('RUN_TRAIN=False; skipping training')

In [ ]:
def latest_checkpoint(checkpoint_dir: Path) -> Path | None:
    candidates = sorted(checkpoint_dir.glob('checkpoint-epoch-*'))
    return candidates[-1] if candidates else None

latest_ckpt = latest_checkpoint(SMOKE_CHECKPOINT_DIR)
print('latest checkpoint:', latest_ckpt)
if latest_ckpt is None:
    raise RuntimeError(f'No checkpoint found under {SMOKE_CHECKPOINT_DIR}')

sample_size = cfg['model']['kwargs']['sample_size']
image_size = int(sample_size[0] if isinstance(sample_size, list) else sample_size)

if RUN_SAMPLE:
    run_cmd([
        PYTHON_BIN,
        'scripts/sample_cosmodiff.py',
        '--checkpoint',
        SMOKE_CHECKPOINT_DIR,
        '--config',
        SMOKE_CONFIG,
        '--output',
        SMOKE_SAMPLE_PATH,
        '--num-samples',
        str(SMOKE_N_GENERATE),
        '--batch-size',
        str(SMOKE_SAMPLE_BATCH),
        '--image-size',
        str(image_size),
        '--seed',
        str(SEED),
        '--device',
        DEVICE,
    ])
else:
    print('RUN_SAMPLE=False; skipping sampling')

In [ ]:
from simdiff_eval.io import _load_real_tanh_from_config, as_nchw
from simdiff_eval.metrics import batch_power_spectra, field_histogram, power_spectrum_summary


def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    arr = np.asarray(arr)
    if limit is None or len(arr) <= limit:
        return np.array(arr, copy=True)
    idx = np.linspace(0, len(arr) - 1, limit, dtype=int)
    return np.array(arr[idx], copy=True)


def load_real_lightweight(config_path: Path, max_raw_samples: int) -> np.ndarray:
    with config_path.open() as f:
        local_cfg = yaml.safe_load(f)
    local_cfg = dict(local_cfg)
    data_cfg = dict(local_cfg.get('data', {}))
    current = data_cfg.get('n_samples')
    data_cfg['n_samples'] = int(max_raw_samples) if current is None else min(int(current), int(max_raw_samples))
    local_cfg['data'] = data_cfg
    return _load_real_tanh_from_config(local_cfg, utils_module=None)


def onepoint_l1(real: np.ndarray, generated: np.ndarray, bins: int = 120) -> tuple[float, dict[str, Any], dict[str, Any]]:
    rh = field_histogram(real, bins=bins)
    gh = field_histogram(generated, bins=bins)
    width = float(np.mean(np.diff(np.asarray(rh['bin_edges']))))
    return float(np.sum(np.abs(np.asarray(rh['hist']) - np.asarray(gh['hist']))) * width), rh, gh

real = load_real_lightweight(SMOKE_CONFIG, max_raw_samples=MAX_REAL_CUBES_FOR_PLOTS)
generated = as_nchw(np.load(SMOKE_SAMPLE_PATH)).copy()

print('real:', real.shape, float(real.mean()), float(real.std()))
print('generated:', generated.shape, float(generated.mean()), float(generated.std()))

In [ ]:
def metric_candidates(checkpoint_dir: Path) -> list[Path]:
    paths: list[Path] = []
    paths.extend(sorted(checkpoint_dir.glob('metrics_epoch_*.json')))
    paths.extend(sorted(checkpoint_dir.glob('metrics.json')))
    for ckpt in sorted(checkpoint_dir.glob('checkpoint-epoch-*')):
        paths.extend(sorted(ckpt.glob('metrics*.json')))
    return paths


def moving_average(x: np.ndarray, window: int) -> np.ndarray:
    if window <= 1 or len(x) < window:
        return x
    kernel = np.ones(window) / window
    return np.convolve(x, kernel, mode='valid')

paths = metric_candidates(SMOKE_CHECKPOINT_DIR)
print('metrics files:', paths[-3:])
metrics = {}
if paths:
    with paths[-1].open() as f:
        metrics = json.load(f)

batch_loss = np.asarray(metrics.get('loss', metrics.get('batch_loss', [])), dtype=float)
epoch_loss = np.asarray(metrics.get('epoch_loss', []), dtype=float)
lr = np.asarray(metrics.get('lr', metrics.get('learning_rate', [])), dtype=float)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
if len(batch_loss):
    window = max(1, len(batch_loss) // 300)
    y = moving_average(batch_loss, window)
    axes[0].plot(np.arange(len(y)), y)
axes[0].set_title('batch loss')
axes[0].set_xlabel('optimizer step')
axes[0].set_ylabel('MSE loss')
axes[0].grid(alpha=0.25)

if len(epoch_loss):
    axes[1].plot(np.arange(len(epoch_loss)), epoch_loss, marker='o')
axes[1].set_title('epoch loss')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('mean MSE loss')
axes[1].grid(alpha=0.25)

if len(lr):
    axes[2].plot(np.arange(len(lr)), lr)
    axes[2].set_yscale('log')
axes[2].set_title('learning rate')
axes[2].set_xlabel('step or epoch')
axes[2].set_ylabel('LR')
axes[2].grid(alpha=0.25)

fig.tight_layout()
out = SMOKE_OUTPUT_DIR / f'{SMOKE_NAME}_loss_curves.png'
fig.savefig(out, dpi=160)
print('wrote', out)

In [ ]:
n_show = min(4, len(generated), len(real))
vals = np.concatenate([real[:n_show].ravel(), generated[:n_show].ravel()])
vmin, vmax = np.nanpercentile(vals, [1, 99])
fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6), squeeze=False)
for i in range(n_show):
    axes[0, i].imshow(real[i, 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
    axes[0, i].set_title(f'real {i}')
    axes[1, i].imshow(generated[i, 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
    axes[1, i].set_title(f'generated {i}')
for ax in axes.ravel():
    ax.axis('off')
fig.suptitle(SMOKE_NAME)
fig.tight_layout()
out = SMOKE_OUTPUT_DIR / f'{SMOKE_NAME}_real_vs_generated_images.png'
fig.savefig(out, dpi=160)
print('wrote', out)

In [ ]:
real_hist_arr = evenly_limit(real, MAX_REAL_HIST)
real_pk_arr = evenly_limit(real, MAX_REAL_PK)
gen_arr = generated

hist_l1, rh, gh = onepoint_l1(real_hist_arr, gen_arr)
pk_summary = power_spectrum_summary(real_pk_arr, gen_arr, nbins=PK_NBINS)

pk_real, kbins = batch_power_spectra(real_pk_arr, nbins=PK_NBINS)
pk_gen, _ = batch_power_spectra(gen_arr, nbins=PK_NBINS)
real_mean = np.nanmean(pk_real, axis=0)
gen_mean = np.nanmean(pk_gen, axis=0)
ratio_pct = 100.0 * (gen_mean - real_mean) / np.clip(real_mean, 1e-30, None)

edges = np.asarray(rh['bin_edges'])
centers = 0.5 * (edges[:-1] + edges[1:])

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].plot(centers, rh['hist'], color='black', label='real')
axes[0].plot(centers, gh['hist'], color='tab:blue', label='generated')
axes[0].set_yscale('log')
axes[0].set_title(f'one-point hist L1={hist_l1:.3f}')
axes[0].set_xlabel('normalized field value')
axes[0].set_ylabel('density')
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].plot(kbins, real_mean, color='black', marker='o', label='real')
axes[1].plot(kbins, gen_mean, color='tab:blue', marker='o', label='generated')
axes[1].set_yscale('log')
axes[1].set_title('mean P(k)')
axes[1].set_xlabel('k bin')
axes[1].set_ylabel('P(k)')
axes[1].grid(alpha=0.25)
axes[1].legend()

axes[2].plot(kbins, ratio_pct, color='tab:blue', marker='o')
axes[2].axhline(0.0, color='black', linestyle=':', linewidth=1.4)
axes[2].set_title(f'P(k) error MAE={pk_summary["pk_log10_mae"]:.3f}')
axes[2].set_xlabel('k bin')
axes[2].set_ylabel('100 * (P_gen - P_real) / P_real [%]')
axes[2].grid(alpha=0.25)

fig.tight_layout()
out = SMOKE_OUTPUT_DIR / f'{SMOKE_NAME}_onepoint_pk.png'
fig.savefig(out, dpi=160)
print('wrote', out)

summary = {
    'smoke_name': SMOKE_NAME,
    'base_run_name': BASE_RUN_NAME,
    'n_real': len(real),
    'n_generated': len(generated),
    'hist_l1': hist_l1,
    'generated_std': gh['std'],
    'real_std': rh['std'],
    'std_ratio': gh['std'] / max(rh['std'], 1e-30),
    **pk_summary,
}
summary_df = pd.DataFrame([summary])
display(summary_df)
summary_path = SMOKE_OUTPUT_DIR / f'{SMOKE_NAME}_summary.csv'
summary_df.to_csv(summary_path, index=False)
print('wrote', summary_path)